<a href="https://colab.research.google.com/github/charansai12322/llmevaluation/blob/main/llmevaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U -q groq

##Study Assistant

In [ ]:
from google.colab import userdata
from groq import Groq

groq_client = Groq(api_key=userdata.get("groq")) # Initialize Groq client


In [ ]:
personalities = {
  "Friendly":  "You are a friendly, enthusiastic, and highly encouraging Study Assistant. Your goal is to break down complex concepts into simple, beginner-friendly explanations. Use analogies and real-world examples that beginners can relate to. Always ask a follow-up question to check understanding",
  "Academic":  "You are a strictly academic, highly detailed, and professional university Professor. Use precise, formal terminology, cite key concepts and structure your response. Your goal is to break down complex concepts into simple, beginner-friendly explanations. Use analogies and real-world examples that beginners can relate to. Always ask a follow-up question to check understanding"
}

def study_assistant(question, persona):
  system_prompt = personalities[persona]

  response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
  )
  return response.choices[0].message.content # Access Groq response correctly

In [ ]:
question = "What are LLMs?"
personality = "Friendly"
assistant_response = study_assistant(question, personality)
print(assistant_response)

**Hey there! 🌟 Let’s dive into what “LLMs” are in a super‑friendly, easy‑to‑understand way.**

---

## 📚 What does “LLM” stand for?
**LLM = Large Language Model**  
Think of it as a **giant digital brain** that’s been trained to understand and generate human language—like English, Spanish, Chinese, and many others.

---

## 🎈 Imagine a Library That Learns

1. **The Library Analogy**  
   - Picture a massive library that contains *every* book, article, website, and chat conversation ever written (well, a huge amount of it).  
   - A librarian (the LLM) reads all these texts, notices patterns, and learns how words and sentences usually go together.

2. **Predicting the Next Word**  
   - When you ask a question or start a sentence, the LLM looks at everything it’s learned and predicts the most likely next words, just like how your phone’s autocomplete suggests the next word while you type a text.

3. **Why “Large”?**  
   - “Large” refers to two things:  
     1. **Size of the model** – 

## LLM as a Judge (Manually)

In [ ]:
evaluation_prompt = """You are an expert evaluator for AI assistants.

Evaluate the Study Assistant's response based on these 6 criteria.
Score each from 1-5 (5 = excellent, 3 = acceptable, 1 = poor).

## Evaluation Criteria:

1. **Accuracy** (1-5): Is the information factually correct?
  - 5: Completely accurate, no errors
  - 3: Mostly accurate, minor issues
  - 1: Contains factual errors

2. **Clarity** (1-5): Is it easy for a beginner (12th class student) to understand?
  - 5: Crystal clear, perfect for beginners
  - 3: Understandable but could be simpler
  - 1: Too complex, uses unexplained jargon

3. **Relevance** (1-5): Does it directly answer the question asked?
  - 5: Perfectly on-topic
  - 3: Mostly relevant with some tangents
  - 1: Goes off-topic or misses the point

4. **Use of Analogies** (1-5): Does it use real-world examples or analogies?
  - 5: Excellent analogies that aid understanding
  - 3: Has some examples but could be better
  - 1: No analogies or examples used

5. **Follow-up Question** (1-5): Does it include a question to check understanding?
  - 5: Thoughtful follow-up question that tests understanding
  - 3: Has a question but it's generic
  - 1: No follow-up question

6. **Persona Consistency** (1-5): Does the tone match the expected persona?
  - For "Friendly": Should be enthusiastic, encouraging, warm
  - For "Academic": Should be formal, precise, professional
  - 5: Perfect match | 3: Somewhat matches | 1: Wrong tone

## Input Details:

**Student Question:** {question}
**Expected Persona:** {persona}
**Assistant Response:** {response}

## Your Evaluation:

Provide your assessment in this exact format:

ACCURACY: [score]/5 - [one line reason]
CLARITY: [score]/5 - [one line reason]
RELEVANCE: [score]/5 - [one line reason]
ANALOGIES: [score]/5 - [one line reason]
FOLLOW_UP: [score]/5 - [one line reason]
PERSONA: [score]/5 - [one line reason]
"""

In [ ]:
from google import genai
from google.colab import userdata

gemini_client = genai.Client(api_key=userdata.get("sec1"))

def evaluate_response(question, persona, response):
    updated_evaluation_prompt = evaluation_prompt.format(
        question=question,
        persona=persona,
        response=response
    )
    evaluation = gemini_client.models.generate_content(
        model="gemini-3.5-flash",
        contents=updated_evaluation_prompt
    )
    return evaluation.text

evaluation_response = evaluate_response(question, personality, assistant_response)
print(evaluation_response)

ACCURACY: 5/5 - The explanation of how LLMs function, including training, parameters, and next-word prediction, is factually correct.
CLARITY: 5/5 - The language is highly accessible, well-structured, and perfectly tailored for a 12th-grade student.
RELEVANCE: 5/5 - The response directly and comprehensively answers the student's question about what LLMs are.
ANALOGIES: 5/5 - It uses outstanding analogies, such as a learning library and phone autocomplete, to explain complex machine learning concepts.
FOLLOW_UP: 5/5 - The question is thoughtful and encourages the student to connect the concept of LLMs to their own daily experiences.
PERSONA: 5/5 - The tone is exceptionally warm, enthusiastic, and encouraging, which perfectly fits the requested "Friendly" persona.


In [ ]:
!pip install -q deepeval

In [ ]:
from deepeval.models import GeminiModel
from deepeval.metrics import AnswerRelevancyMetric

gemini_judge = GeminiModel(
    model="gemini-3.5-flash-lite",
    api_key=userdata.get('sec1')
)

relevance_metric = AnswerRelevancyMetric(
    threshold=0.7,
    model=gemini_judge,
    include_reason=True
)

In [ ]:
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input=question,
    actual_output=assistant_response
)

In [ ]:
from deepeval import evaluate

results = evaluate(
    test_cases=[test_case],
    metrics=[relevance_metric],
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gemini-3.5-flash-lite (Gemini), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=836900;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 57.21s | token cost: 0.0021828000000000004 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

accuracy_metric = GEval(
    name="Accuracy",
    criteria="Determine if the response contains factually correct information about the topic. The explanation should be accurate and free from errors. Students should not learn incorrect concepts.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=gemini_judge,
    threshold=0.7
)

/tmp/ipykernel_2335/2284931752.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


In [ ]:
from deepeval import evaluate

results = evaluate(
    test_cases=[test_case],
    metrics=[relevance_metric],
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gemini-3.5-flash-lite (Gemini), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=597492;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 62.85s | token cost: 0.001563 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.